# 4 
In this notebook I merge members with speeches and clean the data for that to merge well, and add some necessary columns like in_parliament to distinguish guest speakers from politicians. I also assign a gender, using ChatGPT to help identify which Irish names have what gender, as some unique names did not work in the genderguesser.

In [1]:
# Import packages
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from math import factorial
from scipy.stats import multinomial


import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

import matplotlib.pyplot as plt

from unidecode import unidecode

#nltk.download('stopwords')
#nltk.download('wordnet')
#nltk.download('omw-1.4')


In [2]:
df_all_speeches = pd.read_csv('data/merged_speeches_all_mother_and_baby_homes_speeches.csv')
df_members = pd.read_csv('data/members_with_gender_guess.csv')


In [3]:
df_all_speeches.head()

,date,house,debate_uri,speech_number,speaker,member_name,text
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...
1,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...
2,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,291,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...
3,2016-11-17,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,129,Senator Máire Devine,Senator Máire Devine,senator máire devine\n i echo the poi...
4,2016-11-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,116,Deputy Ruth Coppinger,Deputy Ruth Coppinger,deputy ruth coppinger\n the programme...


In [4]:
df_members.head()

,fullName,firstName,lastName,gender,house,constituency,party,role,uri,membership_start,membership_end,gender_guess
0,Henry J. J. Abbott,Henry J. J.,Abbott,NaN,25th Dáil,Longford-Westmeath,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1987-03-10,1989-05-25,male
1,Caroline Acheson,Caroline,Acheson,NaN,22nd Dáil,Tipperary South,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,female
2,Gerry Adams,Gerry,Adams,NaN,32nd Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,male
3,Gerry Adams,Gerry,Adams,NaN,31st Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,male
4,Patrick Agnew,Patrick,Agnew,NaN,22nd Dáil,Louth,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,male


In [5]:
# Basic Preprocessing

import re

# Function to remove speaker name before the first newline
def remove_speaker_name(text):
    if isinstance(text, str):
        # Remove everything before and including the first newline
        return re.sub(r'^.*?\n', '', text)
    return text

# Apply this before lowercasing etc.
df_all_speeches['clean_text'] = df_all_speeches['text'].apply(remove_speaker_name)

# Continue preprocessing steps
df_all_speeches['clean_text'] = df_all_speeches['clean_text'].str.lower()  # lowercase
df_all_speeches['clean_text'] = df_all_speeches['clean_text'].str.replace(r'\n', ' ', regex=True)  # remove \n
df_all_speeches['clean_text'] = df_all_speeches['clean_text'].str.replace(r'[^\w\s]', '', regex=True)  # remove punctuation


# Text length in words
df_all_speeches['text_length'] = df_all_speeches['clean_text'].apply(lambda x: len(x.split()))


In [6]:
df_all_speeches.head()

,date,house,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54
1,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306
2,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,291,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...,the naming issue might seem like a...,146
3,2016-11-17,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,129,Senator Máire Devine,Senator Máire Devine,senator máire devine\n i echo the poi...,i echo the points made by my colleag...,260
4,2016-11-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,116,Deputy Ruth Coppinger,Deputy Ruth Coppinger,deputy ruth coppinger\n the programme...,the programme for government refers ...,157


In [7]:
import re
from unidecode import unidecode

def normalize_speaker(speaker):
    if not isinstance(speaker, str) or not speaker.strip():
        return None
    
    # If there's a name in parentheses, use that (often the real member name)
    paren_match = re.search(r'\((.*?)\)', speaker)
    if paren_match:
        name = paren_match.group(1)
    else:
        name = speaker
    
    # Remove common titles and formal prefixes
    name = re.sub(
        r'\b(Deputy|Senator|Minister(?: of State)?|The Taoiseach|The T[áa]naiste|'
        r'An Ceann Comhairle|An Leas-Cheann Comhairle|An Cathaoirleach|Mr\.?|Ms\.?|Mrs\.?|Dr\.?|'
        r'Professor|Chair|Chairman|Rev|Fr)\b',
        '',
        name,
        flags=re.IGNORECASE
    )

    # Remove role descriptions like “Minister for Children and Youth Affairs”
    name = re.sub(r'Minister [^()]*', '', name, flags=re.IGNORECASE)
    
    # Replace curly quotes and weird apostrophes with straight ones
    name = name.replace("’", "'").replace("`", "'").replace("‘", "'")
    
    # Fix Irish “O ” prefix (e.g. “O Laoghaire” → “O’Laoghaire”)
    name = re.sub(r"\bo\s+([a-z])", r"o'\1", name, flags=re.IGNORECASE)
    
    # Remove punctuation, dots, and compress spaces
    name = re.sub(r"[.,]", "", name)
    name = re.sub(r"\s+", " ", name).strip()
    
    # Normalize accents and lowercase
    name = unidecode(name).lower()
    
    # If after normalization it's empty, return None
    if not name:
        return None

    return name

# Apply to both DataFrames
df_all_speeches['speaker_norm'] = df_all_speeches['speaker'].apply(normalize_speaker)
df_members['fullName_norm'] = df_members['fullName'].apply(
    lambda x: unidecode(x).lower().strip() if isinstance(x, str) else None
)

print(df_all_speeches[['speaker', 'speaker_norm']].head(20))


                                              speaker           speaker_norm
0                           Deputy Maureen O'Sullivan     maureen o'sullivan
1                            Deputy Katherine Zappone      katherine zappone
2                           Deputy Maureen O'Sullivan     maureen o'sullivan
3                                Senator Máire Devine           maire devine
4                               Deputy Ruth Coppinger         ruth coppinger
5                               Senator Victor Boyhan          victor boyhan
6                               Deputy Thomas Pringle         thomas pringle
7                                       The Taoiseach                   None
8   Minister for Children and Youth Affairs (Deput...      katherine zappone
9                            Deputy Katherine Zappone      katherine zappone
10                                  Deputy Clare Daly             clare daly
11                           Deputy Katherine Zappone      katherine zappone

In [8]:

# Manual corrections for specific cases


# Map problematic raw/normalized speaker strings to the correct normalized name
manual_fix_map = {
    # Deputies/Senators that slipped through
    "Deputy Bernard J. Durkan": "bernard durkan",
    "Senator Gerard P. Craughwell": "gerard p. craughwell",
    "Senator David Norris": "david p.b. norris",
    "Deputy Martin Mansergh": "dr martin mansergh",
    "Senator Frank Feighan": "frankie feighan",
    "Deputy Michael P. Kitt": "michael p. kitt",
    "Deputy Paul J. Connaughton": "paul connaughton",
    "Deputy Thomas P. Broughan": "thomas p. broughan",
    "Deputy Frank Feighan": "frankie feighan",
    "Deputy Stephen S. Donnelly": "stephen donnelly",
    "Acting Chairman (Deputy Bernard J. Durkan)": "bernard durkan",
    "Minister for Children and Youth Affairs": "james reilly"
}


manual_overrides = df_all_speeches['speaker'].map(manual_fix_map)

df_all_speeches['speaker_norm'] = manual_overrides.combine_first(df_all_speeches['speaker_norm'])

print("\n--- Normalization After Manual Fixes ---")
print(df_all_speeches[['speaker', 'speaker_norm']].head(20))


missing_after_manual = df_all_speeches[
    ~df_all_speeches['speaker_norm'].isin(df_members['fullName_norm'])
]

print("\n--- Speakers Still Not Matched ---")
if missing_after_manual.empty:
    print("All speakers matched or are guests!")
else:
    print("Raw speaker names not matched:")
    print(missing_after_manual['speaker'].unique())
    print("\nNormalized speaker names not matched:")
    print(missing_after_manual['speaker_norm'].unique())





--- Normalization After Manual Fixes ---
                                              speaker           speaker_norm
0                           Deputy Maureen O'Sullivan     maureen o'sullivan
1                            Deputy Katherine Zappone      katherine zappone
2                           Deputy Maureen O'Sullivan     maureen o'sullivan
3                                Senator Máire Devine           maire devine
4                               Deputy Ruth Coppinger         ruth coppinger
5                               Senator Victor Boyhan          victor boyhan
6                               Deputy Thomas Pringle         thomas pringle
7                                       The Taoiseach                   None
8   Minister for Children and Youth Affairs (Deput...      katherine zappone
9                            Deputy Katherine Zappone      katherine zappone
10                                  Deputy Clare Daly             clare daly
11                           Deput

In [9]:
# Get unique normalized speakers from speeches
speakers_unique = df_all_speeches['speaker_norm'].unique()

# Get normalized member names
members_unique = df_members['fullName_norm'].unique()

# Check which of the 300 are in members
in_members = [s for s in speakers_unique if s in members_unique]
not_in_members = [s for s in speakers_unique if s not in members_unique]

print(f"Speakers found in members: {len(in_members)}")
print(f"Speakers NOT found in members: {len(not_in_members)}")
print("Missing speakers:", not_in_members)

Speakers found in members: 248
Speakers NOT found in members: 66
Missing speakers: [None, 'catherine hynes', 'bernard gloster', 'rosaleen mcdonagh', 'sean o foghlu', 'karen kiernan', 'mary mcdermott', 'simon mcgarr', 'alice coughlan', 'terri harrison', 'lisa kiernan', 'gerry kerr', "maree ryan-o'brien", 'david murphy', 'dale sunderland', 'ann marie flanagan', 'susan lohan', 'patricia carey', 'helen dixon', 'johnny ryan', 'sidney herdman', 'amanda larkin', 'catherine corless', "maeve o'rourke", 'ray murphy', 'doireann ansbro', 'elizabeth carthy', 'david dodd', 'martin parfrey', 'kevin higgins', 'anna corrigan', 'stephen donoghue', 'niamh mccullagh', 'carl buckley', 'maria ni fhlatharta', 'anne mulhall', 'maria corbett', 'seamus mccarthy', 'laura mcgarrigle', "mary lou o'kennedy", "sara o'byrne", 'anna budayova', 'gearoid kenny moore', 'saoirse brady', 'michael macdonagh', 'anna kavanagh', 'catriona crowe', 'fergal lynch', 'joe mccarthy', 'gordon jeyes', 'ruth barrington', 'paul redmond'

In [10]:
df_all_speeches[df_all_speeches["speaker"]=="The Taoiseach"]

,date,house,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm
7,2016-10-26,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,12,The Taoiseach,The Taoiseach,the taoiseach\n i know the deputy has...,i know the deputy has raised this ou...,338,None
20,2016-02-02,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,283,The Taoiseach,The Taoiseach,the taoiseach\n i would like to say a...,i would like to say a few words on t...,1145,None
29,2020-12-08,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,65,The Taoiseach,The Taoiseach,"the taoiseach\n i know the minister, ...",i know the minister deputy roderic o...,200,None
56,2020-11-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,230,The Taoiseach,The Taoiseach,the taoiseach\n in regard to deputy...,in regard to deputy boyd barretts ...,928,None
62,2020-11-11,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,99,The Taoiseach,The Taoiseach,the taoiseach\n the government has ma...,the government has made it clear tha...,93,None
...,...,...,...,...,...,...,...,...,...,...
2069,2018-09-25,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,149,The Taoiseach,The Taoiseach,the taoiseach\n i am sorry to tell ...,i am sorry to tell the deputy that...,180,None
2102,2018-05-30,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,107,The Taoiseach,The Taoiseach,the taoiseach\n on the audit of recor...,on the audit of records i answered t...,557,None
2108,2018-05-22,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,155,The Taoiseach,The Taoiseach,the taoiseach\n it is a survey that...,it is a survey that is designed to...,380,None
2114,2018-05-01,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,2,The Taoiseach,The Taoiseach,the taoiseach\n i say once again how ...,i say once again how saddened i was ...,717,None


In [11]:
# Role-Holder Lookup DataFrame
# from wikipedia
role_data = [
    # Role, Start Date, End Date, Normalized Full Name
    ('The Taoiseach', '2008-05-07', '2011-03-09', 'brian cowen'),
    ('The Taoiseach', '2011-03-09', '2017-06-14', 'enda kenny'),
    ('The Taoiseach', '2017-06-14', '2020-06-27', 'leo varadkar'),
    ('The Taoiseach', '2020-06-27', '2022-12-17', 'micheal martin'),
    ('The Taoiseach', '2022-12-17', '2024-04-09', 'leo varadkar'),
    ('The Taoiseach', '2024-04-09', '2025-01-23', 'simon harris'),
    ('The Taoiseach', '2025-01-23', None, 'micheal martin'), # Example of a current role
    ('An Leas-Cheann Comhairle', '2007-06-26', '2011-03-09', 'brendan howlin'),
    ('An Leas-Cheann Comhairle', '2011-03-31', '2016-03-10', 'michael p. kitt'),
    ('An Leas-Cheann Comhairle', '2016-07-07', '2020-01-14', 'pat the cope gallagher'),
    ('An Leas-Cheann Comhairle', '2020-07-23', '2024-11-08', 'catherine connolly'),
    ('An Leas-Cheann Comhairle', '2025-02-19', None, 'john mcguiness'),
    ('An Cathaoirleach', '2007-09-13', '2011-05-25', 'pat moylan'),
    ('An Cathaoirleach', '2011-05-25', '2016-06-08', 'paddy burke'),
    ('An Cathaoirleach', '2016-06-08', '2020-06-29', "denis o'donovan"),
    ('An Cathaoirleach', '2020-06-29', '2022-12-16', "mark daly"),
    ('An Cathaoirleach', '2022-12-16', '2024-11-30', "jerry buttimer"),
    ('An Cathaoirleach', '2025-02-12', None, "mark daly"),
    ('The Tánaiste', '2008-05-07', '2011-03-09', 'mary coughlan'),
    ('The Tánaiste', '2011-03-09', '2014-07-04', 'eamon gilmore'),
    ('The Tánaiste', '2014-07-04', '2016-05-06', 'joan burton'),
    ('The Tánaiste', '2016-05-06', '2017-11-28', 'frances fitzgerald'),
    ('The Tánaiste', '2017-11-30', '2020-06-27', 'simon coveney'),
    ('The Tánaiste', '2020-06-27', '2022-12-17', 'leo varadkar'),
    ('The Tánaiste', '2022-12-17', '2025-01-23', 'micheal martin'),
    ('The Tánaiste', '2025-01-23', None, 'simon harris'),
    ('An Ceann Comhairle', '2007-06-14', '2009-10-13', "john o'donoghue"),
    ('An Ceann Comhairle', '2009-10-13', '2011-03-09', 'seamus kirk'),
    ('An Ceann Comhairle', '2011-03-09', '2016-03-10', 'sean barrett'),
    ('An Ceann Comhairle', '2016-03-10', '2024-12-18', 'sean o fearghail'),
    ('An Ceann Comhairle', '2024-12-18', None, 'verona murphy'),
    ('Chairman', '2007-06-14', '2009-10-13', "john o'donoghue"),
    ('Chairman', '2009-10-13', '2011-03-09', 'seamus kirk'),
    ('Chairman', '2011-03-09', '2016-03-10', 'sean barrett'),
    ('Chairman', '2016-03-10', '2024-12-18', 'sean o fearghail'),
    ('Chairman', '2024-12-18', None, 'verona murphy')
]
role_lookup = pd.DataFrame(role_data, columns=['role', 'start_date', 'end_date', 'fullName_norm'])

print("--- Role Lookup Before Conversion ---")
print(role_lookup.dtypes)

# Convert role_lookup dates
role_lookup['start_date'] = pd.to_datetime(role_lookup['start_date'])
# For end_date, we fill any missing (None/NaT) values with a future date
# This ensures that people still in the role are correctly matched
role_lookup['end_date'] = pd.to_datetime(role_lookup['end_date']).fillna(pd.Timestamp('2099-12-31'))

print("\n--- Role Lookup After Conversion ---")
print(role_lookup.dtypes)
role_lookup.head()



--- Role Lookup Before Conversion ---
role             object
start_date       object
end_date         object
fullName_norm    object
dtype: object

--- Role Lookup After Conversion ---
role                     object
start_date       datetime64[ns]
end_date         datetime64[ns]
fullName_norm            object
dtype: object


,role,start_date,end_date,fullName_norm
0,The Taoiseach,2008-05-07,2011-03-09,brian cowen
1,The Taoiseach,2011-03-09,2017-06-14,enda kenny
2,The Taoiseach,2017-06-14,2020-06-27,leo varadkar
3,The Taoiseach,2020-06-27,2022-12-17,micheal martin
4,The Taoiseach,2022-12-17,2024-04-09,leo varadkar


In [12]:
df_all_speeches['date'] = pd.to_datetime(df_all_speeches['date'])

def find_role_holder_at(role, when):
    candidates = role_lookup[role_lookup['role'] == role]
    mask = (candidates['start_date'] <= when) & (when <= candidates['end_date'])
    match = candidates.loc[mask]
    if not match.empty:
        return match['fullName_norm'].iat[0]
    return None

# apply to create a new column with the taoiseach at the time of the speech
df_all_speeches['taoiseach_at_time'] = df_all_speeches['date'].apply(lambda d: find_role_holder_at('The Taoiseach', d))

df_all_speeches


,date,house,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm,taoiseach_at_time
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,enda kenny
1,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306,katherine zappone,enda kenny
2,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,291,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...,the naming issue might seem like a...,146,maureen o'sullivan,enda kenny
3,2016-11-17,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,129,Senator Máire Devine,Senator Máire Devine,senator máire devine\n i echo the poi...,i echo the points made by my colleag...,260,maire devine,enda kenny
4,2016-11-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,116,Deputy Ruth Coppinger,Deputy Ruth Coppinger,deputy ruth coppinger\n the programme...,the programme for government refers ...,157,ruth coppinger,enda kenny
...,...,...,...,...,...,...,...,...,...,...,...
2226,2013-02-27,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,455,Senator Susan O'Keeffe,Senator Susan O'Keeffe,senator susan o'keeffe\n i welcome th...,i welcome the minister of state depu...,1086,susan o'keeffe,enda kenny
2227,2013-02-26,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,276,Deputy Brian Walsh,Deputy Brian Walsh,deputy brian walsh\n i welcome the op...,i welcome the opportunity to contrib...,871,brian walsh,enda kenny
2228,2013-02-26,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,277,Deputy Marcella Corcoran Kennedy,Deputy Marcella Corcoran Kennedy,deputy marcella corcoran kennedy\n it...,it is with profound acknowledgement ...,1131,marcella corcoran kennedy,enda kenny
2229,2013-02-19,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,300,Deputy Mick Wallace,Deputy Mick Wallace,deputy mick wallace\n more than a yea...,more than a year ago i was approache...,672,mick wallace,enda kenny


In [13]:
# If 'speaker_norm' is None/NaN and speaker is 'The Taoiseach', fill with taoiseach_at_time
mask = (df_all_speeches['speaker'] == 'The Taoiseach') & (df_all_speeches['speaker_norm'].isna())
df_all_speeches.loc[mask, 'speaker_norm'] = df_all_speeches.loc[mask, 'taoiseach_at_time']


In [14]:
df_all_speeches[df_all_speeches['speaker'] == 'The Taoiseach']['speaker_norm'].unique()


array(['enda kenny', 'micheal martin', 'leo varadkar'], dtype=object)

now do the rest of the official people roles ... and then after that merge with member data on speaker_norm

In [15]:
# Fill speaker_norm for other roles (excluding The Taoiseach)
other_roles = role_lookup['role'].unique()
other_roles = [r for r in other_roles if r != 'The Taoiseach']

for role in other_roles:
    mask = (df_all_speeches['speaker'] == role) & (df_all_speeches['speaker_norm'].isna())
    df_all_speeches.loc[mask, 'speaker_norm'] = df_all_speeches.loc[mask, 'date'].apply(lambda d: find_role_holder_at(role, d))


In [16]:
df_all_speeches

,date,house,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm,taoiseach_at_time
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,enda kenny
1,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306,katherine zappone,enda kenny
2,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,291,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...,the naming issue might seem like a...,146,maureen o'sullivan,enda kenny
3,2016-11-17,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,129,Senator Máire Devine,Senator Máire Devine,senator máire devine\n i echo the poi...,i echo the points made by my colleag...,260,maire devine,enda kenny
4,2016-11-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,116,Deputy Ruth Coppinger,Deputy Ruth Coppinger,deputy ruth coppinger\n the programme...,the programme for government refers ...,157,ruth coppinger,enda kenny
...,...,...,...,...,...,...,...,...,...,...,...
2226,2013-02-27,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,455,Senator Susan O'Keeffe,Senator Susan O'Keeffe,senator susan o'keeffe\n i welcome th...,i welcome the minister of state depu...,1086,susan o'keeffe,enda kenny
2227,2013-02-26,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,276,Deputy Brian Walsh,Deputy Brian Walsh,deputy brian walsh\n i welcome the op...,i welcome the opportunity to contrib...,871,brian walsh,enda kenny
2228,2013-02-26,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,277,Deputy Marcella Corcoran Kennedy,Deputy Marcella Corcoran Kennedy,deputy marcella corcoran kennedy\n it...,it is with profound acknowledgement ...,1131,marcella corcoran kennedy,enda kenny
2229,2013-02-19,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,300,Deputy Mick Wallace,Deputy Mick Wallace,deputy mick wallace\n more than a yea...,more than a year ago i was approache...,672,mick wallace,enda kenny


In [17]:
df_members

,fullName,firstName,lastName,gender,house,constituency,party,role,uri,membership_start,membership_end,gender_guess,fullName_norm
0,Henry J. J. Abbott,Henry J. J.,Abbott,NaN,25th Dáil,Longford-Westmeath,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1987-03-10,1989-05-25,male,henry j. j. abbott
1,Caroline Acheson,Caroline,Acheson,NaN,22nd Dáil,Tipperary South,Fianna Fáil,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,female,caroline acheson
2,Gerry Adams,Gerry,Adams,NaN,32nd Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,male,gerry adams
3,Gerry Adams,Gerry,Adams,NaN,31st Dáil,Louth,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,male,gerry adams
4,Patrick Agnew,Patrick,Agnew,NaN,22nd Dáil,Louth,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1981-06-30,1982-02-27,male,patrick agnew
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7215,Aengus Ó Snodaigh,Aengus,Ó Snodaigh,NaN,33rd Dáil,Dublin South-Central,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-02-08,2024-11-08,male,aengus o snodaigh
7216,Aengus Ó Snodaigh,Aengus,Ó Snodaigh,NaN,34th Dáil,Dublin South-Central,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2024-11-29,NaN,male,aengus o snodaigh
7217,Fionntán Ó Súilleabháin,Fionntán,Ó Súilleabháin,NaN,34th Dáil,Wicklow-Wexford,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2024-11-29,NaN,male,fionntan o suilleabhain
7218,Eineachán Ó hAnnluain,Eineachán,Ó hAnnluain,NaN,16th Dáil,Monaghan,Sinn Féin,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,1957-03-20,1961-09-15,unknown,eineachan o hannluain


In [18]:
# Merge speeches with members on normalized names
df_merged = df_all_speeches.merge(
    df_members,
    how='left',
    left_on='speaker_norm',
    right_on='fullName_norm',
    indicator=True  # adds a column showing merge status
)

# Rows that didn't match (speakers not found in members)
missing_speakers = df_merged[df_merged['_merge'] == 'left_only']['speaker'].unique()

print("Speakers not found in members API:")
for s in missing_speakers:
    print("-", s)




Speakers not found in members API:
- Ms Catherine Hynes
- Mr. Bernard Gloster
- Dr. Rosaleen McDonagh
- Mr. Seán Ó Foghlú
- Ms Karen Kiernan
- Ms Mary McDermott
- Mr. Simon McGarr
- Ms Alice Coughlan
- Ms Terri Harrison
- Ms Lisa Kiernan
- Mr. Gerry Kerr
- Ms Maree Ryan-O'Brien
- Mr. David Murphy
- Mr. Dale Sunderland
- Ms Ann Marie Flanagan
- Ms Susan Lohan
- Ms Patricia Carey
- Ms Helen Dixon
- Dr. Johnny Ryan
- Mr. Sidney Herdman
- Ms Amanda Larkin
- Ms Catherine Corless
- Dr. Maeve O'Rourke
- Professor Ray Murphy
- Ms Doireann Ansbro
- Ms Elizabeth Carthy
- Mr. David Dodd
- Mr. Martin Parfrey
- Mr. Kevin Higgins
- Ms Anna Corrigan
- Dr. Stephen Donoghue
- Dr. Niamh McCullagh
- Mr. Carl Buckley
- Ms Maria Ní Fhlatharta
- Ms Anne Mulhall
- Ms Maria Corbett
- Mr. Seamus McCarthy
- Ms Laura McGarrigle
- Ms Mary Lou O'Kennedy
- Dr. Sara O'Byrne
- Ms Anna Budayova
- Mr. Gearóid Kenny Moore
- Ms Saoirse Brady
- Mr. Michael MacDonagh
- Ms Anna Kavanagh
- Ms Catriona Crowe
- Dr. Fergal Lync

In [19]:
df_merged['in_parliament'] = df_merged['_merge'].map({
    'both': 'yes',
    'left_only': 'no',
    'right_only': 'unknown'  # should rarely occur with left join
})

In [20]:
df_merged.columns

Index(['date', 'house_x', 'debate_uri', 'speech_number', 'speaker',
       'member_name', 'text', 'clean_text', 'text_length', 'speaker_norm',
       'taoiseach_at_time', 'fullName', 'firstName', 'lastName', 'gender',
       'house_y', 'constituency', 'party', 'role', 'uri', 'membership_start',
       'membership_end', 'gender_guess', 'fullName_norm', '_merge',
       'in_parliament'],
      dtype='object')

In [21]:
df_merged['in_parliament'].value_counts()

in_parliament
yes        7371
no          116
unknown       0
Name: count, dtype: int64

In [22]:
#pd.set_option('display.max_columns', None)
df_merged.head()

,date,house_x,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm,...,constituency,party,role,uri,membership_start,membership_end,gender_guess,fullName_norm,_merge,in_parliament
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,maureen o'sullivan,both,yes
1,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
2,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2009-06-05,2011-02-01,female,maureen o'sullivan,both,yes
3,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306,katherine zappone,...,Dublin South-West,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,katherine zappone,both,yes
4,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,288,Deputy Katherine Zappone,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306,katherine zappone,...,Nominated by the Taoiseach,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-05-25,2016-04-24,female,katherine zappone,both,yes


In [23]:
df_merged['gender_guess'].value_counts()

gender_guess
male       3710
female     2721
unknown     940
Name: count, dtype: int64

In [24]:
missing_gender = df_members[df_members['gender_guess']== 'unknown']

In [25]:
# Get unique full names with unknown gender
missing_gender_names = df_members[df_members['gender_guess'] == 'unknown']['fullName_norm'].unique()

print(missing_gender_names)


['ciaran ahern' 'lorcan allen' 'moosajee bhamjee' 'paraic brady'
 'pat breen' 'dr. martin brennan' 'pat buckley' 'ulick burke'
 'toddie byrne' 'jackie cahill' 'mairia cahill' 'phelim alfred calleary'
 'ciaran cannon' 'pat carey' 'micheal carrigy' 'pat casey' 'donie cassidy'
 'sorca clarke' 'prof. arthur edward clery' 'paudie coffey'
 'paudge connolly' 'pat cox' 'prof. james craig' 'reada cronin'
 'ollie crowe' 'dr. john crowley' 'ciaran cuffe' 'pa daly' 'pat deering'
 'countess of desart' 'pearse doherty' 'paschal donohoe' 'dr. ada english'
 'olwyn enright' 'osmond t. grattan esmonde' 'nugent talbot everard'
 'pat farrell' 'padraig faulkner' 'frankie feighan' 'pat gallagher'
 'pat the cope gallagher' 'roisin garvey' 'captain patrick giles'
 'lord glenavy' 'camillus glynn' 'dr maurice hayes' 'jackie healy-rae'
 'carmencita hederman' 'dr. thomas hennessy' 'alice-mary higgins'
 'tras honan' 'dr. francis humphreys' 'hon andrew jameson' 'carey joyce'
 'c. gordon lambert' 'pat lee' 'kearns l

In [26]:
#pd.set_option('display.max_rows', None)

df_merged['fullName_norm'].value_counts()

fullName_norm
enda kenny            442
micheal martin        396
mary lou mcdonald     304
catherine connolly    285
ivana bacik           258
                     ... 
joan freeman            1
steven matthews         1
kevin o'keeffe          1
michael mullins         1
mary moran              1
Name: count, Length: 255, dtype: int64

In [27]:
df_merged['fullName_norm'].isna().sum()

np.int64(116)

In [28]:
#guest speakers who aren't in the members_df will have nothing in fullName_norm of course, so i will impute their speaker_norm values into fullName_norm column as I want to be able to use fullName_norm as a non-missing value column for analysis
df_merged['fullName_norm'] = df_merged['fullName_norm'].fillna(df_merged['speaker_norm'])

In [29]:
df_merged['fullName_norm'].isna().sum()

np.int64(0)

In [30]:
# Chatgpt

# Dictionary of missing parliament gendered names → gender guesses ('male', 'female', None if unsure)
missing_gender_dict = {
    'ciaran ahern': 'male',
    'lorcan allen': 'male',
    'moosajee bhamjee': 'male',
    'paraic brady': 'male',
    'pat breen': 'male',
    'dr. martin brennan': 'male',
    'pat buckley': 'male',
    'ulick burke': 'male',
    'toddie byrne': 'male',
    'jackie cahill': 'male',
    'mairia cahill': 'female',
    'phelim alfred calleary': 'male',
    'ciaran cannon': 'male',
    'pat carey': 'male',
    'micheal carrigy': 'male',
    'pat casey': 'male',
    'donie cassidy': 'male',
    'sorca clarke': 'female',
    'prof. arthur edward clery': 'male',
    'paudie coffey': 'male',
    'paudge connolly': 'male',
    'pat cox': 'male',
    'prof. james craig': 'male',
    'reada cronin': 'female',
    'ollie crowe': 'male',
    'dr. john crowley': 'male',
    'ciaran cuffe': 'male',
    'pa daly': 'male',
    'pat deering': 'male',
    'countess of desart': 'female',
    'pearse doherty': 'male',
    'paschal donohoe': 'male',
    'dr. ada english': 'female',
    'olwyn enright': 'female',
    'osmond t. grattan esmonde': 'male',
    'nugent talbot everard': 'male',
    'pat farrell': 'male',
    'padraig faulkner': 'male',
    'frankie feighan': 'male',
    'pat gallagher': 'male',
    'pat the cope gallagher': 'male',
    'roisin garvey': 'female',
    'captain patrick giles': 'male',
    'lord glenavy': 'male',
    'camillus glynn': 'male',
    'dr maurice hayes': 'male',
    'jackie healy-rae': 'male',
    'carmencita hederman': 'female',
    'dr. thomas hennessy': 'male',
    'alice-mary higgins': 'female',
    'tras honan': 'male',
    'dr. francis humphreys': 'male',
    'hon andrew jameson': 'male',
    'carey joyce': 'male',
    'c. gordon lambert': 'male',
    'pat lee': 'male',
    'kearns linda kearns': 'female',
    'fionan lynch': 'male',
    'dr. kathleen lynn': 'female',
    'professor william magennis': 'male',
    'b. j. maguire': 'male',
    'dr martin mansergh': 'male',
    'micheal martin': 'male',
    'aubrey mccarthy': 'male',
    'dr. patrick mccarvill': 'male',
    'jarlath mcdonagh': 'male',
    'dinny mcginley': 'male',
    'finian mcgrath': 'male',
    'capt. sydney b. minch': 'male',
    'paschal mooney': 'male',
    'pat moylan': 'male',
    'ronan mullen': 'male',
    'pj murphy': 'male',
    'ged nash': 'male',
    'm. j. nolan': 'male',
    'evanne ni chuilinn': 'female',
    'shonagh ni raghallaigh': 'female',
    "marie-louise o'donnell": 'female',
    "peader o'donnell": 'male',
    "dr. patrick joseph o'dowd": 'male',
    "malachai o'hara": 'male',
    "dr thomas francis o'higgins snr.": 'male',
    "batt o'keeffe": 'male',
    "donogh o'malley": 'male',
    "pat o'neill": 'male',
    "prof. john marcus o'sullivan": 'male',
    "toddy o'sullivan": 'male',
    'averil power': 'female',
    'pat rabbitte': 'male',
    'mcgillycuddy of the reeks': 'male',
    'brid rogers': 'female',
    'j n ross': 'male',
    'dr. robert james rowlette': 'male',
    'col. jeremiah ryan': 'male',
    'p. j. sheehan': 'male',
    'roisin shortall': 'female',
    'prof. william f. p. stockley': 'male',
    'prof. william e. thrift': 'male',
    'prof. michael tierney': 'male',
    'dr. sean tubridy': 'male',
    'pat upton': 'male',
    'dr. f.c. ward': 'male',
    'dr. vincent joseph white': 'male',
    'ollie wilkinson': 'male',
    'hutcheson william hutcheson': 'male',
    'g. v. wright': 'male',
    'windham wyndham-quin': 'male',
    'violet-anne wynne': 'female',
    'pearse wyse': 'male',
    'deirdre de burca': 'female',
    'vivion de valera': 'male',
    'caoimhghin o caolain': 'male',
    'donall o conallain': 'male',
    'eamon o cuiv': 'male',
    'pol o foighil': 'male',
    'labhras o murchu': 'male',
    'eineachan o hannluain': 'male'
}


In [31]:
# Fill chatgpt gender_guess in df_merged
df_merged['gender_guess'] = df_merged.apply(
    lambda row: missing_gender_dict.get(row['fullName_norm'], row['gender_guess']),
    axis=1
)

# Verify
missing_after = df_merged['gender_guess'].isna().sum()
print(f"Number of rows with missing gender_guess after filling: {missing_after}")

Number of rows with missing gender_guess after filling: 116


In [32]:
#is it the guest speakers who are missing the gender_guess?
missing_guests = df_merged.loc[
    (df_merged['in_parliament'] == 'no') & (df_merged['gender_guess'].isna())
].shape[0]

print("Missing gender_guess for guest speakers:", missing_guests)

#yes!

Missing gender_guess for guest speakers: 116


In [33]:
# Rows where gender_guess is still missing
still_missing = df_merged[df_merged['gender_guess'].isna()]

# Show unique names with missing gender
missing_names = still_missing['speaker_norm'].unique()
print("Names still missing gender_guess:")
print(missing_names)

# Optional: see how many rows are missing
print("\nTotal rows with missing gender_guess:", len(still_missing))


Names still missing gender_guess:
['catherine hynes' 'bernard gloster' 'rosaleen mcdonagh' 'sean o foghlu'
 'karen kiernan' 'mary mcdermott' 'simon mcgarr' 'alice coughlan'
 'terri harrison' 'lisa kiernan' 'gerry kerr' "maree ryan-o'brien"
 'david murphy' 'dale sunderland' 'ann marie flanagan' 'susan lohan'
 'patricia carey' 'helen dixon' 'johnny ryan' 'sidney herdman'
 'amanda larkin' 'catherine corless' "maeve o'rourke" 'ray murphy'
 'doireann ansbro' 'elizabeth carthy' 'david dodd' 'martin parfrey'
 'kevin higgins' 'anna corrigan' 'stephen donoghue' 'niamh mccullagh'
 'carl buckley' 'maria ni fhlatharta' 'anne mulhall' 'maria corbett'
 'seamus mccarthy' 'laura mcgarrigle' "mary lou o'kennedy" "sara o'byrne"
 'anna budayova' 'gearoid kenny moore' 'saoirse brady' 'michael macdonagh'
 'anna kavanagh' 'catriona crowe' 'fergal lynch' 'joe mccarthy'
 'gordon jeyes' 'ruth barrington' 'paul redmond' 'kathy mcmahon'
 'mary slattery' 'chris fitzpatrick' 'geoffrey shannon' 'eilionoir flynn'
 '

In [34]:
# Assign gender guesses for guest speakers (from Chatgpt)
guest_gender_dict = {
    'catherine hynes': 'female',
    'bernard gloster': 'male',
    'rosaleen mcdonagh': 'female',
    'sean o foghlu': 'male',
    'karen kiernan': 'female',
    'mary mcdermott': 'female',
    'simon mcgarr': 'male',
    'alice coughlan': 'female',
    'terri harrison': 'female',
    'lisa kiernan': 'female',
    'gerry kerr': 'male',
    'maree ryan-o\'brien': 'female',
    'david murphy': 'male',
    'dale sunderland': 'male',
    'ann marie flanagan': 'female',
    'susan lohan': 'female',
    'patricia carey': 'female',
    'helen dixon': 'female',
    'johnny ryan': 'male',
    'sidney herdman': 'male',
    'amanda larkin': 'female',
    'catherine corless': 'female',
    'maeve o\'rourke': 'female',
    'ray murphy': 'male',
    'doireann ansbro': 'female',
    'elizabeth carthy': 'female',
    'david dodd': 'male',
    'martin parfrey': 'male',
    'kevin higgins': 'male',
    'anna corrigan': 'female',
    'stephen donoghue': 'male',
    'niamh mccullagh': 'female',
    'carl buckley': 'male',
    'maria ni fhlatharta': 'female',
    'anne mulhall': 'female',
    'maria corbett': 'female',
    'seamus mccarthy': 'male',
    'laura mcgarrigle': 'female',
    'mary lou o\'kennedy': 'female',
    'sara o\'byrne': 'female',
    'anna budayova': 'female',
    'gearoid kenny moore': 'male',
    'saoirse brady': 'female',
    'michael macdonagh': 'male',
    'anna kavanagh': 'female',
    'catriona crowe': 'female',
    'fergal lynch': 'male',
    'joe mccarthy': 'male',
    'gordon jeyes': 'male',
    'ruth barrington': 'female',
    'paul redmond': 'male',
    'kathy mcmahon': 'female',
    'mary slattery': 'female',
    'chris fitzpatrick': 'male',
    'geoffrey shannon': 'male',
    'eilionoir flynn': 'female',
    'nem kearns': 'female',
    'aoife conduit': 'female',
    'carmel mcdonnell byrne': 'female',
    'patrick rodgers': 'male',
    'conor falvey': 'male',
    'peter gohery': 'male',
    'aoiveen mathews': 'female',
    'kevin mccarthy': 'male',
    'maria corbett': 'female',
    'michael walsh': 'male'
}

# Fill gender_guess for guest speakers
df_merged['gender_guess'] = df_merged.apply(
    lambda row: guest_gender_dict.get(row['fullName_norm'], row['gender_guess']),
    axis=1
)

# Verify
missing_after = df_merged['gender_guess'].isna().sum()
print(f"Rows still missing gender_guess: {missing_after}")


Rows still missing gender_guess: 0


In [35]:
df_merged.columns

Index(['date', 'house_x', 'debate_uri', 'speech_number', 'speaker',
       'member_name', 'text', 'clean_text', 'text_length', 'speaker_norm',
       'taoiseach_at_time', 'fullName', 'firstName', 'lastName', 'gender',
       'house_y', 'constituency', 'party', 'role', 'uri', 'membership_start',
       'membership_end', 'gender_guess', 'fullName_norm', '_merge',
       'in_parliament'],
      dtype='object')

In [36]:
df_merged[['gender_guess', 'fullName_norm', '_merge',
       'in_parliament']]

,gender_guess,fullName_norm,_merge,in_parliament
0,female,maureen o'sullivan,both,yes
1,female,maureen o'sullivan,both,yes
2,female,maureen o'sullivan,both,yes
3,female,katherine zappone,both,yes
4,female,katherine zappone,both,yes
...,...,...,...,...
7482,male,michael p. kitt,both,yes
7483,male,michael p. kitt,both,yes
7484,male,michael p. kitt,both,yes
7485,male,michael p. kitt,both,yes


when merging, all the times a member of parliament was in office merged with their speech (which means there are a lot of duplicates as a lot of members were in office for various rounds). So, get rid of duplicates now by making sure the date of the speech is within the membership_start and membership_end dates

In [37]:
print(df_merged['date'].dtype)
print(df_merged['membership_start'].dtype)
print(df_merged['membership_end'].dtype)

datetime64[ns]
object
object


In [38]:
import pandas as pd

# Ensure datetime objects
df_merged['date'] = pd.to_datetime(df_merged['date'])
df_merged['membership_start'] = pd.to_datetime(df_merged['membership_start'])
df_merged['membership_end'] = pd.to_datetime(df_merged['membership_end'], errors='coerce')

# Handle active members (Fill NaT in end date with today)
# (Crucial for current politicians so the <= comparison works)
df_merged['membership_end'] = df_merged['membership_end'].fillna(pd.Timestamp.now())

# Condition A: It is a valid term (Speech date is inside the membership range)
term_match_mask = (df_merged['date'] >= df_merged['membership_start']) & \
                  (df_merged['date'] <= df_merged['membership_end'])

# Condition B: It is a guest speaker (They have no term dates, so we keep them)
# We treat rows with 'no' in 'in_parliament' OR rows where membership_start is missing as guests
guest_mask = (df_merged['in_parliament'] == 'no') | (df_merged['membership_start'].isna())

# Keep row if (Dates Match) OR (Is Guest)
df_merged_unique = df_merged[term_match_mask | guest_mask].copy()

print(f"Original rows: {len(df_merged)}")
print(f"Filtered rows: {len(df_merged_unique)}")
print("\nSample of cleaned data:")
print(df_merged_unique[['date', 'speaker', 'membership_start', 'membership_end']].head())

Original rows: 7487
Filtered rows: 2224

Sample of cleaned data:
         date                    speaker membership_start membership_end
0  2016-12-14  Deputy Maureen O'Sullivan       2016-03-10     2020-01-14
3  2016-12-14   Deputy Katherine Zappone       2016-03-10     2020-01-14
5  2016-12-14  Deputy Maureen O'Sullivan       2016-03-10     2020-01-14
8  2016-11-17       Senator Máire Devine       2016-04-25     2020-03-29
10 2016-11-17      Deputy Ruth Coppinger       2016-03-10     2020-01-14


In [39]:
df_merged_unique[df_merged_unique['speaker_norm'] == "maureen o'sullivan"]

,date,house_x,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm,...,constituency,party,role,uri,membership_start,membership_end,gender_guess,fullName_norm,_merge,in_parliament
0,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,287,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,maureen o'sullivan,both,yes
5,2016-12-14,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,291,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...,the naming issue might seem like a...,146,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,maureen o'sullivan,both,yes
79,2016-02-02,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,301,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n tá sé doc...,tá sé dochreidte go bhfuil an díospó...,816,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
911,2017-06-01,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,180,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n we have h...,we have had so many inquiries report...,699,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,maureen o'sullivan,both,yes
1382,2017-03-09,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,11,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n we meet s...,we meet so many people individuals a...,602,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2016-03-10,2020-01-14,female,maureen o'sullivan,both,yes
3123,2014-11-18,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,85,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question concerns those ladies ...,53,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
3377,2014-07-17,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,241,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n comhghair...,comhghairdeas minister and my very b...,853,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
3477,2014-07-01,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,37,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question is to ask the minister...,23,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
3488,2014-07-01,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,39,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n i welco...,i welcome the ministers statement ...,233,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes
3499,2014-07-01,dail,https://data.oireachtas.ie/akn/ie/debateRecord...,41,Deputy Maureen O'Sullivan,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the iri...,the irish human rights commission ...,171,maureen o'sullivan,...,Dublin Central,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2011-03-09,2016-03-09,female,maureen o'sullivan,both,yes

In [40]:
# Define the columns that should make a row unique
unique_cols = ['date', 'speaker_norm', 'text_length']

# Find all rows that have a duplicate (keep=False shows all involved rows)
duplicate_rows = df_merged_unique[
    df_merged_unique.duplicated(subset=unique_cols, keep=False)
]

if duplicate_rows.empty:
    print("Great! No duplicates found based on that criteria.")
else:
    print(f"Found {len(duplicate_rows)} rows that are part of a duplicate set.")
    
    # Sort them to see the duplicates next to each other
    print("\n--- Sample of Duplicate Rows ---")
    print(duplicate_rows.sort_values(by=unique_cols).head(10))

Found 4 rows that are part of a duplicate set.

--- Sample of Duplicate Rows ---
           date house_x                                         debate_uri  \
6214 2023-04-19  seanad  https://data.oireachtas.ie/akn/ie/debateRecord...   
6231 2023-04-19  seanad  https://data.oireachtas.ie/akn/ie/debateRecord...   
6197 2023-05-16  seanad  https://data.oireachtas.ie/akn/ie/debateRecord...   
6202 2023-05-16  seanad  https://data.oireachtas.ie/akn/ie/debateRecord...   

      speech_number                                            speaker  \
6214             75  Minister for Children, Equality, Disability, I...   
6231             94  Minister for Children, Equality, Disability, I...   
6197            126                              Senator Sharon Keogan   
6202            138                              Senator Sharon Keogan   

                                            member_name  \
6214  Minister for Children, Equality, Disability, I...   
6231  Minister for Children, Equality, 

In [41]:
duplicate_rows

,date,house_x,debate_uri,speech_number,speaker,member_name,text,clean_text,text_length,speaker_norm,...,constituency,party,role,uri,membership_start,membership_end,gender_guess,fullName_norm,_merge,in_parliament
6197,2023-05-16,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,126,Senator Sharon Keogan,Senator Sharon Keogan,senator sharon keogan\n having looked...,having looked after children who had...,89,sharon keogan,...,Industrial and Commercial Panel,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-03-30,2025-01-29,female,sharon keogan,both,yes
6202,2023-05-16,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,138,Senator Sharon Keogan,Senator Sharon Keogan,senator sharon keogan\n i am well awa...,i am well aware who it was intended ...,89,sharon keogan,...,Industrial and Commercial Panel,Independent,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-03-30,2025-01-29,female,sharon keogan,both,yes
6214,2023-04-19,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,75,"Minister for Children, Equality, Disability, I...","Minister for Children, Equality, Disability, I...","minister for children, equality, disability, i...",today i am bringing the mother and b...,2459,roderic o'gorman,...,Dublin West,Green Party,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-02-08,2024-11-08,male,roderic o'gorman,both,yes
6231,2023-04-19,seanad,https://data.oireachtas.ie/akn/ie/debateRecord...,94,"Minister for Children, Equality, Disability, I...","Minister for Children, Equality, Disability, I...","minister for children, equality, disability, i...",i thank the senators for their detai...,2459,roderic o'gorman,...,Dublin West,Green Party,TD or Senator,https://data.oireachtas.ie/ie/oireachtas/membe...,2020-02-08,2024-11-08,male,roderic o'gorman,both,yes


It worked!

In [42]:
len(df_merged_unique)

2224

df_merged_clean:
- are speaker and member_name the exact same? Yes - then only keep speaker for df_merged_clean. what about 'fullName', 'firstName', 'lastName'...
- get rid of gender column as it is empty
- get rid of both uri columns and speech_number 
- get rid of '_merge' as we have 'in_parliament'
- speaker_norm and fullName_norm are the same so only keep one...
- role has only the value TD or Senator and as we already have a in_parliament column we can drop it



In [43]:
df_merged_clean = df_merged_unique.drop(columns=['debate_uri', 'speech_number', 'member_name', 'gender', 'uri', '_merge', 'fullName_norm', 'role']).copy()

In [44]:

df_merged_clean.head()

,date,house_x,speaker,text,clean_text,text_length,speaker_norm,taoiseach_at_time,fullName,firstName,lastName,house_y,constituency,party,membership_start,membership_end,gender_guess,in_parliament
0,2016-12-14,dail,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n my ques...,my question relates to a request f...,54,maureen o'sullivan,enda kenny,Maureen O'Sullivan,Maureen,O'Sullivan,32nd Dáil,Dublin Central,Independent,2016-03-10,2020-01-14,female,yes
3,2016-12-14,dail,Deputy Katherine Zappone,deputy katherine zappone\n i am awa...,i am aware of the request by the g...,306,katherine zappone,enda kenny,Katherine Zappone,Katherine,Zappone,32nd Dáil,Dublin South-West,Independent,2016-03-10,2020-01-14,female,yes
5,2016-12-14,dail,Deputy Maureen O'Sullivan,deputy maureen o'sullivan\n the nam...,the naming issue might seem like a...,146,maureen o'sullivan,enda kenny,Maureen O'Sullivan,Maureen,O'Sullivan,32nd Dáil,Dublin Central,Independent,2016-03-10,2020-01-14,female,yes
8,2016-11-17,seanad,Senator Máire Devine,senator máire devine\n i echo the poi...,i echo the points made by my colleag...,260,maire devine,enda kenny,Máire Devine,Máire,Devine,25th Seanad,Labour Panel,Sinn Féin,2016-04-25,2020-03-29,female,yes
10,2016-11-17,dail,Deputy Ruth Coppinger,deputy ruth coppinger\n the programme...,the programme for government refers ...,157,ruth coppinger,enda kenny,Ruth Coppinger,Ruth,Coppinger,32nd Dáil,Dublin West,Anti-Austerity Alliance - People Before Profit,2016-03-10,2020-01-14,female,yes


Add a binary column of "in_government" to determine which parties were part of the coalition/the government

In [45]:
# Ensure the date column is in datetime format
df_merged_clean['date'] = pd.to_datetime(df_merged_clean['date'])

def is_in_government(row):
    date = row['date']
    party = row['party']
    
    # 2007–2011: Fianna Fáil, Green Party, Progressive Democrats
    if pd.Timestamp('2007-06-14') <= date <= pd.Timestamp('2011-03-08'):
        return 1 if party in ['Fianna Fáil', 'Green Party', 'Progressive Democrats'] else 0
    
    # 2011–2016: Fine Gael, Labour
    elif pd.Timestamp('2011-03-09') <= date <= pd.Timestamp('2016-05-05'):
        return 1 if party in ['Fine Gael', 'Labour Party'] else 0
    
    # 2016–2020: Fine Gael, Independents (Minority)
    elif pd.Timestamp('2016-05-06') <= date <= pd.Timestamp('2020-06-26'):
        return 1 if party in ['Fine Gael', 'Independent'] else 0
    
    # 2020–2024: Fianna Fáil, Fine Gael, Green Party
    elif pd.Timestamp('2020-06-27') <= date <= pd.Timestamp('2025-01-22'):
        return 1 if party in ['Fianna Fáil', 'Fine Gael', 'Green Party'] else 0
    
    # 2025–Present: Fianna Fáil, Fine Gael, Independents
    elif date >= pd.Timestamp('2025-01-23'):
        return 1 if party in ['Fianna Fáil', 'Fine Gael', 'Independent'] else 0
    
    return 0

# Apply the function to create the new column
df_merged_clean['in_government'] = df_merged_clean.apply(is_in_government, axis=1)

#so 1 is in government, and 0 is in opposition.

df_merged_clean.to_csv("data/df_merged_clean.csv", index=False) #for R STM


In [46]:
print(df_merged_clean['in_government'].value_counts())

in_government
0    1384
1     840
Name: count, dtype: int64


In [47]:
df_merged_clean.to_csv("data/df_merged_clean.csv", index=False)